# Modelos lineales con scikit-learn - Precios de Propiedades en CABA

Implementación de modelos lineales con sklearn. Agrega one-hot encoding de `barrio` y `comuna` a las features numéricas
y compara `LinearRegression`, `Ridge` y `Lasso` contra el baseline de los notebooks anteriores.

**Features (X)**: `metros`, `ambientes`, `banos`, `expensas` + one-hot de `barrio` y `comuna`  
**Target (y)**: `precio` en USD

## Estructura
1. Setup e imports
2. Datos (carga, split, preprocesamiento)
3. Modelos lineales (`LinearRegression`, `Ridge`, `Lasso`)
4. Evaluación y comparación

### 1. Setup/Imports

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

### 2. Datos

Mismo split 70/15/15 que los notebooks anteriores. El preprocesamiento incluye one-hot encoding de `barrio` y `comuna`:
se crea una columna binaria por categoría y se dropea una (`drop='first'`) para evitar multicolinealidad.
El `ColumnTransformer` aplica z-score a las features numéricas y one-hot a las categóricas en un solo paso.

In [11]:
df = pd.read_csv("../data/processed/zonaprop_clean.csv")

np.random.seed(42)
idx = np.random.permutation(len(df))
df = df.iloc[idx].reset_index(drop=True)
n = len(df)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train = df.iloc[:n_train]
val   = df.iloc[n_train:n_train+n_val]
test  = df.iloc[n_train+n_val:]

num_features = ["metros", "ambientes", "banos", "expensas"]
cat_features = ["barrio", "comuna"]
target = "precio"

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features),
])

X_train, y_train = train[num_features + cat_features], train[target].values
X_val,   y_val   = val[num_features + cat_features],   val[target].values
X_test,  y_test  = test[num_features + cat_features],  test[target].values

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Barrios únicos: {df['barrio'].nunique()} | Comunas únicas: {df['comuna'].nunique()}")

Train: (22809, 6) | Val: (4887, 6) | Test: (4889, 6)
Barrios únicos: 42 | Comunas únicas: 15


### 3. Modelos lineales

Cada modelo se arma como un `Pipeline` que encadena el preprocesador con el estimador.
Esto garantiza que el scaling y el one-hot se fittean solo sobre train y se aplican a val y test.

In [12]:
models = {
    "LinearRegression": Pipeline([("pre", preprocessor), ("model", LinearRegression())]),
    "Ridge":            Pipeline([("pre", preprocessor), ("model", Ridge())]),
    "Lasso":            Pipeline([("pre", preprocessor), ("model", Lasso(max_iter=10_000))]),
}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"{name} entrenado")

LinearRegression entrenado
Ridge entrenado
Lasso entrenado


### 4. Evaluación y comparación

Comparamos los tres modelos entre sí y contra el baseline de regresión lineal sin `barrio` (notebook 04, RMSE=167,361, R²=0.5379).

In [13]:
print(f"{'Modelo':<20} {'RMSE':>12}  {'MAE':>12}  {'R²':>6}")
print("-" * 56)

test_results = {}
for name, pipe in models.items():
    y_hat = pipe.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_hat)
    mae  = mean_absolute_error(y_test, y_hat)
    r2   = r2_score(y_test, y_hat)
    test_results[name] = {"RMSE": rmse, "MAE": mae, "R²": r2}
    print(f"{name:<20} {rmse:>12,.0f}  {mae:>12,.0f}  {r2:>6.4f}")

print("-" * 56)
print(f"{'Baseline (nb04)':<20} {'167,361':>12}  {'96,184':>12}  {'0.5379':>6}")

Modelo                       RMSE           MAE      R²
--------------------------------------------------------
LinearRegression          145,960        88,134  0.6485
Ridge                     145,952        88,118  0.6485
Lasso                     145,958        88,130  0.6485
--------------------------------------------------------
Baseline (nb04)           167,361        96,184  0.5379
